# Loan Eligibility Predictions

**Import libs**

In [15]:
# Main libraries for data manipulation
import pandas as pd
import numpy as np

# Visualization libraries
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as sp

# Machine learning libraries
from sklearn.model_selection import train_test_split

from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score

# preprocessing and encoding
from sklearn.preprocessing import LabelEncoder, OneHotEncoder
from sklearn.pipeline import Pipeline

import joblib

In [2]:
loan_data = pd.read_csv('../data/Loan_Eligibility_Prediction.csv')

In [3]:
# quick look at the data
display(loan_data.head())

print("=" * 40)

display(loan_data.info())
print("=" * 40)
display(loan_data.describe(include='all'))
print("=" * 40)
display(loan_data.isnull().sum())

,Customer_ID,Gender,Married,Dependents,Education,Self_Employed,Applicant_Income,Coapplicant_Income,Loan_Amount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,569,Female,No,0,Graduate,No,2378,0.0,9,360,1,Urban,N
1,15,Male,Yes,2,Graduate,No,1299,1086.0,17,120,1,Urban,Y
2,95,Male,No,0,Not Graduate,No,3620,0.0,25,120,1,Semiurban,Y
3,134,Male,Yes,0,Graduate,Yes,3459,0.0,25,120,1,Semiurban,Y
4,556,Male,Yes,1,Graduate,No,5468,1032.0,26,360,1,Semiurban,Y


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 614 entries, 0 to 613
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Customer_ID         614 non-null    int64  
 1   Gender              614 non-null    object 
 2   Married             614 non-null    object 
 3   Dependents          614 non-null    int64  
 4   Education           614 non-null    object 
 5   Self_Employed       614 non-null    object 
 6   Applicant_Income    614 non-null    int64  
 7   Coapplicant_Income  614 non-null    float64
 8   Loan_Amount         614 non-null    int64  
 9   Loan_Amount_Term    614 non-null    int64  
 10  Credit_History      614 non-null    int64  
 11  Property_Area       614 non-null    object 
 12  Loan_Status         614 non-null    object 
dtypes: float64(1), int64(6), object(6)
memory usage: 62.5+ KB


None

,Customer_ID,Gender,Married,Dependents,Education,Self_Employed,Applicant_Income,Coapplicant_Income,Loan_Amount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
count,614.000000,614,614,614.000000,614,614,614.000000,614.000000,614.000000,614.000000,614.000000,614,614
unique,NaN,2,2,NaN,2,2,NaN,NaN,NaN,NaN,NaN,3,2
top,NaN,Male,Yes,NaN,Graduate,No,NaN,NaN,NaN,NaN,NaN,Semiurban,Y
freq,NaN,499,399,NaN,480,523,NaN,NaN,NaN,NaN,NaN,233,422
mean,307.500000,NaN,NaN,0.856678,NaN,NaN,5403.459283,1621.245798,142.022801,338.892508,0.850163,NaN,NaN
std,177.390811,NaN,NaN,1.216651,NaN,NaN,6109.041673,2926.248369,87.083089,69.716355,0.357203,NaN,NaN
min,1.000000,NaN,NaN,0.000000,NaN,NaN,150.000000,0.000000,9.000000,12.000000,0.000000,NaN,NaN
25%,154.250000,NaN,NaN,0.000000,NaN,NaN,2877.500000,0.000000,98.000000,360.000000,1.000000,NaN,NaN
50%,307.500000,NaN,NaN,0.000000,NaN,NaN,3812.500000,1188.500000,125.000000,360.000000,1.000000,NaN,NaN
75%,460.750000,NaN,NaN,2.000000,NaN,NaN,5795.000000,2297.250000,164.750000,360.000000,1.000000,NaN,NaN


Customer_ID           0
Gender                0
Married               0
Dependents            0
Education             0
Self_Employed         0
Applicant_Income      0
Coapplicant_Income    0
Loan_Amount           0
Loan_Amount_Term      0
Credit_History        0
Property_Area         0
Loan_Status           0
dtype: int64

In [4]:
# Display the corelation between the features


fig = sp.make_subplots(rows=2, cols=2, subplot_titles=[
    'Gender vs Loan Status',
    'Married vs Loan Status',
    'Credit History vs Loan Status',
    'Property Area vs Loan Status'
])

fig.add_trace(
    go.Bar(
        x=loan_data['Gender'].value_counts().index,
        y=loan_data['Gender'].value_counts().values,
        name='Gender vs Loan Status',
        marker_color='indianred'
    ),
    row=1, col=1
)

fig.add_trace(
    go.Bar(
        x=loan_data['Married'].value_counts().index,
        y=loan_data['Married'].value_counts().values,
        name='Married vs Loan Status',
        marker_color='lightsalmon'
    ),
    row=1, col=2
)

fig.add_trace(
    go.Bar(
        x=loan_data['Credit_History'].value_counts().index,
        y=loan_data['Credit_History'].value_counts().values,
        name='Credit History vs Loan Status',
        marker_color='lightseagreen'
    ),
    row=2, col=1
)

fig.add_trace(
    go.Bar(
        x=loan_data['Property_Area'].value_counts().index,
        y=loan_data['Property_Area'].value_counts().values,
        name='Property Area vs Loan Status',
        marker_color='mediumpurple',
    ),
    row=2, col=2
)

fig.update_layout(height=900, width=900, title_text="Categorical Features vs Loan Status")
fig.show()

# COmapring which gender is more likely to get loan approved
fig = px.histogram(loan_data, x='Gender', color='Loan_Status', barmode='group',
                   title='Gender vs Loan Status', color_discrete_sequence=px.colors.qualitative.Pastel, template='plotly_dark')
fig.show()

## Key observations

- Male applicants submit more applications than female applicants.
- Approval is strongly associated with credit history; applicants with positive credit history are approved at substantially higher rates.
- Married applicants have higher approval rates than single applicants.
- Applicants from semiurban areas receive approvals more frequently than those from rural areas.
- Credit score and verified income are the primary determinants of loan approval; demographic attributes are secondary.

## **Model Creation, Prediction and Comparations**

In [5]:
# Encoding

def encode_data(df):
    df.dropna(inplace=True)
    df.drop(columns=['Customer_ID'], inplace=True)

    # Encoding categorical variables
    le = LabelEncoder()
    le_cols = ['Gender', 'Married', 'Education', 'Self_Employed', 'Property_Area', 'Loan_Status']
    for col in le_cols:
        df[col] = le.fit_transform(df[col])

    return df

loan_enc = encode_data(loan_data)
loan_enc.head()

,Gender,Married,Dependents,Education,Self_Employed,Applicant_Income,Coapplicant_Income,Loan_Amount,Loan_Amount_Term,Credit_History,Property_Area,Loan_Status
0,0,0,0,0,0,2378,0.0,9,360,1,2,0
1,1,1,2,0,0,1299,1086.0,17,120,1,2,1
2,1,0,0,1,0,3620,0.0,25,120,1,1,1
3,1,1,0,0,1,3459,0.0,25,120,1,1,1
4,1,1,1,0,0,5468,1032.0,26,360,1,1,1


In [6]:
# Feature engineering

loan_enc['Total_Income'] = loan_enc['Applicant_Income'] + loan_enc['Coapplicant_Income']

In [7]:
# Train test split

X = loan_enc.drop(columns=['Loan_Status'])
y = loan_enc['Loan_Status']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=67
)

In [19]:
# Data Scaling
X_scale_train = X_train.copy()
X_scale_test = X_test.copy()
y_scale_train = y_train.copy()
y_scale_test = y_test.copy()
# Logistic Regression

scaler = StandardScaler()
X_scale_train = scaler.fit_transform(X_scale_train)
X_scale_test = scaler.transform(X_scale_test)

log_reg = LogisticRegression(C=1, random_state=67)
log_reg.fit(X_scale_train, y_scale_train)
y_pred = log_reg.predict(X_scale_test)

scoreboard = {
    'Model': [],
    'Precision': [],
    'Recall': [],
    'F1 Score': []
}

scoreboard['Model'].append('Logistic Regression (C=1)')
report = classification_report(y_scale_test, y_pred, output_dict=True)
scoreboard['F1 Score'].append(f1_score(y_scale_test, y_pred))
scoreboard['Precision'].append(report['1']['precision'])
scoreboard['Recall'].append(report['1']['recall'])

In [20]:
# Decision Tree Classifier
dt_clf = DecisionTreeClassifier(max_depth=4, random_state=67)
dt_clf.fit(X_train, y_train)
y_pred = dt_clf.predict(X_test)
    
scoreboard['Model'].append('Decision Tree Classifier (max_depth=4)')
report = classification_report(y_test, y_pred, output_dict=True)
scoreboard['F1 Score'].append(f1_score(y_test, y_pred))
scoreboard['Precision'].append(report['1']['precision'])
scoreboard['Recall'].append(report['1']['recall'])

In [21]:
# Random Forest Classifier
rf_clf = RandomForestClassifier(n_estimators=100, max_depth=4, random_state=67)
rf_clf.fit(X_train, y_train)
y_pred = rf_clf.predict(X_test)

scoreboard['Model'].append('Random Forest Classifier (n_estimators=100, max_depth=4)')
report = classification_report(y_test, y_pred, output_dict=True)
scoreboard['F1 Score'].append(f1_score(y_test, y_pred))
scoreboard['Precision'].append(report['1']['precision'])
scoreboard['Recall'].append(report['1']['recall'])

In [23]:
scoreboard_df = pd.DataFrame(scoreboard)
display(scoreboard_df)

# Compare models 

px.bar(scoreboard_df, x='Model', y=['Precision', 'Recall', 'F1 Score'], barmode='group', text_auto=True,
       title='Model Comparison: Precision, Recall and F1 Score', color_discrete_sequence=px.colors.qualitative.Pastel, template='plotly_dark').show()

,Model,Precision,Recall,F1 Score
0,Logistic Regression (C=1),0.796117,0.964706,0.872340
1,Decision Tree Classifier (max_depth=4),0.788462,0.964706,0.867725
2,"Random Forest Classifier (n_estimators=100, ma...",0.796117,0.964706,0.872340


### __Best model__: Logistic Regression and Random Forest

In [24]:
# for ease of use, lets use the Random Forest Classifier as our final model
final_model = rf_clf

def predict_loan_eligibility(input_data, model=final_model):

    encode_data(input_data)
    input_data['Total_Income'] = input_data['Applicant_Income'] + input_data['Coapplicant_Income']
    """
    Predict loan eligibility using the trained model.

    Parameters:
    model: Trained machine learning model (default is Random Forest Classifier)
    input_data: DataFrame containing input features for prediction

    Returns:
    predictions: Array of predicted loan eligibility
    """
    predictions = model.predict(input_data)
    return predictions